# SQLite - The Complete Professional Handbook (Interactive Notebook)

**Kajola Gbenga**
Data Scientist & AI Engineer

*Companion executable notebook to the SQLite handbook - the fifth and final RDBMS in the SQL learning sequence (after PostgreSQL, MySQL, SQL Server, and Oracle).*

Run all cells top to bottom. Each section builds its own tables inside a local `handbook.db` file (recreated fresh in the first cell), so the notebook is fully reproducible from a clean state.

In [ ]:
import sqlite3, os

DB_PATH = "handbook.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()
print("SQLite library version:", sqlite3.sqlite_version)
print("Python sqlite3 module version:", sqlite3.version)

## 1. Architecture Check - PRAGMAs That Reveal Internals

In [ ]:
for pragma in ["page_size", "encoding", "journal_mode", "foreign_keys", "user_version", "application_id"]:
    val = conn.execute(f"PRAGMA {pragma};").fetchone()[0]
    print(f"{pragma:15s} -> {val}")

## 2. Dynamic Typing and Type Affinity (Part 6 of the handbook)

Demonstrates that a `NUMERIC` affinity column can store four *different* storage classes across rows in the same column - impossible in PostgreSQL/MySQL/SQL Server/Oracle without an explicit error.

In [ ]:
conn.execute("DROP TABLE IF EXISTS demo")
conn.execute("CREATE TABLE demo (val NUMERIC)")
conn.executemany("INSERT INTO demo VALUES (?)", [(10,), (10.5,), ('10.5',), ('hello',), (None,)])
conn.commit()

for row in conn.execute("SELECT val, typeof(val) FROM demo"):
    print(row["val"], "->", row["typeof(val)"] if "typeof(val)" in row.keys() else row[1])

## 3. STRICT Tables (Part 7) - SQLite >= 3.37 required

If your SQLite build is older than 3.37, this cell will raise a syntax error on `STRICT` - that itself demonstrates the version dependency called out in the handbook.

In [ ]:
conn.execute("DROP TABLE IF EXISTS products_strict")
try:
    conn.execute('''
        CREATE TABLE products_strict (
            product_id INTEGER PRIMARY KEY,
            product_name TEXT NOT NULL,
            price REAL NOT NULL
        ) STRICT;
    ''')
    conn.execute("INSERT INTO products_strict VALUES (1, 'Widget', 9.99)")
    conn.commit()
    print("STRICT table created and inserted successfully.")
    try:
        conn.execute("INSERT INTO products_strict VALUES (2, 'Bad Price', 'nine ninety nine')")
        conn.commit()
    except sqlite3.IntegrityError as e:
        print("Expected failure on bad type insert:", e)
except sqlite3.OperationalError as e:
    print("STRICT not supported on this SQLite build:", e)

## 4. ROWID and INTEGER PRIMARY KEY (Part 8)

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS orders_demo;
CREATE TABLE orders_demo (
    order_id INTEGER PRIMARY KEY,   -- alias for ROWID
    customer_id INTEGER,
    total REAL
);
''')
conn.execute("INSERT INTO orders_demo (customer_id, total) VALUES (1, 99.5)")
conn.commit()
row = conn.execute("SELECT order_id, ROWID AS rid FROM orders_demo").fetchone()
print("order_id:", row["order_id"], "| ROWID:", row["rid"], "-> they are the same value")

## 5. Constraints and ON CONFLICT Strategies (Part 10)

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS tags;
CREATE TABLE tags (
    tag_id INTEGER PRIMARY KEY,
    name TEXT UNIQUE ON CONFLICT IGNORE
);
''')
conn.execute("INSERT INTO tags (name) VALUES ('python')")
conn.execute("INSERT INTO tags (name) VALUES ('python')")  # silently ignored
conn.commit()
print(conn.execute("SELECT * FROM tags").fetchall())

## 6. UPSERT - INSERT OR REPLACE vs ON CONFLICT DO UPDATE (Parts 10.2, 13)

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE,
    loyalty_points INTEGER DEFAULT 0
);
''')
conn.execute("INSERT INTO customers VALUES (1, 'Ada Lovelace', 'ada@example.com', 500)")
conn.commit()

# INSERT OR REPLACE: loses loyalty_points (resets to DEFAULT) because it is delete+insert
conn.execute("INSERT OR REPLACE INTO customers (customer_id, name, email) VALUES (1, 'Ada L.', 'ada@example.com')")
conn.commit()
print("After INSERT OR REPLACE:", dict(conn.execute("SELECT * FROM customers WHERE customer_id=1").fetchone()))

conn.execute("UPDATE customers SET loyalty_points = 500 WHERE customer_id = 1")
conn.commit()

# True UPSERT preserves loyalty_points
conn.execute('''
INSERT INTO customers (customer_id, name, email) VALUES (1, 'Ada Lovelace', 'ada@example.com')
ON CONFLICT(customer_id) DO UPDATE SET name = excluded.name
''')
conn.commit()
print("After ON CONFLICT DO UPDATE:", dict(conn.execute("SELECT * FROM customers WHERE customer_id=1").fetchone()))

## 7. Joins, CTEs, Recursive CTEs, and Window Functions (Parts 18-22)

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS departments; DROP TABLE IF EXISTS employees;
CREATE TABLE departments (department_id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY, name TEXT, manager_id INTEGER,
    department_id INTEGER, salary REAL
);
INSERT INTO departments VALUES (1,'Engineering'), (2,'Sales');
INSERT INTO employees VALUES
 (1,'Chinedu Okafor', NULL, 1, 250000),
 (2,'Amara Nwosu', 1, 1, 180000),
 (3,'Tunde Bakare', 1, 1, 170000),
 (4,'Zainab Bello', NULL, 2, 200000),
 (5,'Yusuf Sani', 4, 2, 120000);
''')
conn.commit()

print("--- JOIN ---")
for r in conn.execute('''
    SELECT e.name, d.name AS dept FROM employees e JOIN departments d ON d.department_id=e.department_id
'''):
    print(dict(r))

print("\n--- Recursive CTE: org chart ---")
for r in conn.execute('''
WITH RECURSIVE org(employee_id, name, manager_id, level) AS (
    SELECT employee_id, name, manager_id, 0 FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.employee_id, e.name, e.manager_id, o.level+1
    FROM employees e JOIN org o ON e.manager_id = o.employee_id
)
SELECT * FROM org ORDER BY level, name
'''):
    print(dict(r))

print("\n--- Window function: salary rank within department ---")
for r in conn.execute('''
    SELECT name, department_id, salary,
           RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rnk
    FROM employees
'''):
    print(dict(r))

## 8. Indexes and Query Plan Diagnosis (Parts 25-26)

In [ ]:
conn.execute("DROP TABLE IF EXISTS orders_big")
conn.executescript('''
CREATE TABLE orders_big (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, total REAL);
''')
import random
from datetime import date, timedelta
rows = [(cid, (date(2026,1,1)+timedelta(days=random.randint(0,300))).isoformat(), round(random.uniform(10,500),2))
        for cid in [random.randint(1,50) for _ in range(3000)]]
conn.executemany("INSERT INTO orders_big (customer_id, order_date, total) VALUES (?,?,?)", rows)
conn.commit()

print("--- BEFORE index ---")
for r in conn.execute("EXPLAIN QUERY PLAN SELECT * FROM orders_big WHERE customer_id = 7 ORDER BY order_date"):
    print(dict(r))

conn.execute("CREATE INDEX idx_orders_big_cust_date ON orders_big(customer_id, order_date)")
conn.commit()

print("\n--- AFTER index ---")
for r in conn.execute("EXPLAIN QUERY PLAN SELECT * FROM orders_big WHERE customer_id = 7 ORDER BY order_date"):
    print(dict(r))

## 9. Transactions, Savepoints, and WAL Mode (Parts 28-29)

In [ ]:
conn.execute("PRAGMA journal_mode = WAL;")
print("journal_mode now:", conn.execute("PRAGMA journal_mode;").fetchone()[0])

conn.executescript('''
DROP TABLE IF EXISTS accounts;
CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, balance REAL);
INSERT INTO accounts VALUES (1, 1000), (2, 500);
''')
conn.commit()

conn.execute("BEGIN")
conn.execute("UPDATE accounts SET balance = balance - 100 WHERE account_id = 1")
conn.execute("SAVEPOINT before_credit")
conn.execute("UPDATE accounts SET balance = balance + 100 WHERE account_id = 2")
conn.execute("ROLLBACK TO before_credit")   # undo only the credit
conn.execute("RELEASE before_credit")
conn.commit()

print(conn.execute("SELECT * FROM accounts").fetchall())
print("Notice account 2's balance is unchanged - only the debit from account 1 was kept.")

## 10. JSON Functions and Indexed Generated Columns (Parts 38-39)

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS customer_profile;
CREATE TABLE customer_profile (
    customer_id INTEGER PRIMARY KEY,
    profile JSON NOT NULL,
    country TEXT GENERATED ALWAYS AS (json_extract(profile, '$.country')) VIRTUAL,
    risk_score REAL GENERATED ALWAYS AS (json_extract(profile, '$.risk_score')) STORED
);
''')
conn.execute('''INSERT INTO customer_profile (customer_id, profile) VALUES
(1, ?), (2, ?)''', (
    '{"country":"NG","age":29,"subscription":"premium","risk_score":0.12}',
    '{"country":"GH","age":41,"subscription":"basic","risk_score":0.83}'
))
conn.commit()
conn.execute("CREATE INDEX IF NOT EXISTS idx_profile_risk ON customer_profile(risk_score)")

for r in conn.execute("SELECT customer_id, country, risk_score FROM customer_profile WHERE risk_score > 0.5"):
    print(dict(r))

## 11. Full-Text Search with FTS5 (Part 37)

If your SQLite build was not compiled with FTS5, this cell will raise an error - that itself demonstrates the version/build dependency called out in the handbook.

In [ ]:
try:
    conn.executescript('''
        DROP TABLE IF EXISTS documents_fts;
        CREATE VIRTUAL TABLE documents_fts USING fts5(title, content);
    ''')
    conn.executemany("INSERT INTO documents_fts (title, content) VALUES (?, ?)", [
        ("SQLite Basics", "SQLite is a self-contained embedded relational database engine."),
        ("Indexing Guide", "Indexes speed up search operations using B-trees."),
        ("Fraud Detection", "Detecting fraud requires analyzing transaction patterns over time."),
    ])
    conn.commit()

    print("--- MATCH search ---")
    for r in conn.execute("SELECT title FROM documents_fts WHERE documents_fts MATCH 'database'"):
        print(dict(r))

    print("\n--- Ranked with bm25 ---")
    for r in conn.execute('''
        SELECT title, bm25(documents_fts) AS score FROM documents_fts
        WHERE documents_fts MATCH 'search' ORDER BY score
    '''):
        print(dict(r))
except sqlite3.OperationalError as e:
    print("FTS5 not available on this SQLite build:", e)

## 12. Triggers - Audit Log Example (Part 40)

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS order_audit_log;
CREATE TABLE order_audit_log (
    log_id INTEGER PRIMARY KEY, order_id INTEGER, old_total REAL, new_total REAL, changed_at TEXT
);
DROP TRIGGER IF EXISTS trg_orders_audit;
CREATE TRIGGER trg_orders_audit AFTER UPDATE ON orders_demo
BEGIN
    INSERT INTO order_audit_log (order_id, old_total, new_total, changed_at)
    VALUES (old.order_id, old.total, new.total, datetime('now'));
END;
''')
conn.commit()

conn.execute("UPDATE orders_demo SET total = 149.99 WHERE order_id = 1")
conn.commit()
print(conn.execute("SELECT * FROM order_audit_log").fetchall())

## 13. Python SQLite CRUD Project - Student Management System (Part 44)

In [ ]:
from contextlib import contextmanager

SCHEMA = '''
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name  TEXT NOT NULL,
    email      TEXT UNIQUE NOT NULL,
    grade      INTEGER NOT NULL CHECK(grade BETWEEN 1 AND 12),
    gpa        REAL DEFAULT 0.0,
    enrolled_at TEXT DEFAULT CURRENT_TIMESTAMP
);
'''

@contextmanager
def get_connection():
    c = sqlite3.connect(DB_PATH)
    c.execute("PRAGMA foreign_keys = ON;")
    c.row_factory = sqlite3.Row
    try:
        yield c
        c.commit()
    except Exception:
        c.rollback()
        raise
    finally:
        c.close()

def init_db():
    with get_connection() as c:
        c.executescript(SCHEMA)

def create_student(full_name, email, grade, gpa=0.0):
    with get_connection() as c:
        cur = c.execute(
            "INSERT INTO students (full_name, email, grade, gpa) VALUES (?, ?, ?, ?) RETURNING student_id",
            (full_name, email, grade, gpa),
        )
        return cur.fetchone()["student_id"]

def read_student(student_id):
    with get_connection() as c:
        row = c.execute("SELECT * FROM students WHERE student_id = ?", (student_id,)).fetchone()
        return dict(row) if row else None

def update_student(student_id, **fields):
    if not fields:
        return False
    columns = ", ".join(f"{k} = ?" for k in fields)
    values = list(fields.values()) + [student_id]
    with get_connection() as c:
        cur = c.execute(f"UPDATE students SET {columns} WHERE student_id = ?", values)
        return cur.rowcount > 0

def delete_student(student_id):
    with get_connection() as c:
        cur = c.execute("DELETE FROM students WHERE student_id = ?", (student_id,))
        return cur.rowcount > 0

def search_students(keyword):
    with get_connection() as c:
        cur = c.execute(
            "SELECT * FROM students WHERE full_name LIKE ? OR email LIKE ?",
            (f"%{keyword}%", f"%{keyword}%"),
        )
        return [dict(r) for r in cur.fetchall()]

init_db()
sid = create_student("Kajola Gbenga", "kajola@example.com", 10, 3.8)
print("Created:", read_student(sid))
update_student(sid, gpa=3.95)
print("Updated:", read_student(sid))
print("Search:", search_students("Kajola"))
delete_student(sid)
print("After delete:", read_student(sid))

## 14. SQLite with pandas (Part 45)

In [ ]:
import pandas as pd

df = pd.read_sql("SELECT * FROM orders_big LIMIT 5", conn)
display(df) if 'display' in globals() else print(df)

monthly_revenue = pd.read_sql('''
    SELECT strftime('%Y-%m', order_date) AS month, ROUND(SUM(total),2) AS revenue
    FROM orders_big GROUP BY month ORDER BY month
''', conn)
print(monthly_revenue)

monthly_revenue.to_sql("monthly_revenue_report", conn, if_exists="replace", index=False)
print("\nWritten back to SQLite as 'monthly_revenue_report' table.")

## 15. SQLite with SQLAlchemy (Part 46)

Requires the `sqlalchemy` package. If it is not installed in your environment, install it first with `pip install sqlalchemy`.

In [ ]:
try:
    from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
    from sqlalchemy.orm import declarative_base, sessionmaker, relationship

    engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
    Base = declarative_base()

    class CustomerORM(Base):
        __tablename__ = "customers_orm"
        customer_id = Column(Integer, primary_key=True)
        name = Column(String, nullable=False)
        email = Column(String, unique=True)

    Base.metadata.create_all(engine)
    Session = sessionmaker(bind=engine)
    session = Session()

    session.add(CustomerORM(name="Bola Ade", email="bola@example.com"))
    session.commit()

    results = session.query(CustomerORM).filter(CustomerORM.name.like("%Ade%")).all()
    for r in results:
        print(r.customer_id, r.name, r.email)
    session.close()
except ImportError:
    print("SQLAlchemy is not installed in this environment. Run: pip install sqlalchemy")

## 16. Sales Analytics Database Project (Part 68)

In [ ]:
import random
from datetime import date, timedelta

conn.executescript('''
DROP TABLE IF EXISTS transactions_sales; DROP TABLE IF EXISTS products_sales; DROP TABLE IF EXISTS customers_sales;
CREATE TABLE customers_sales (customer_id INTEGER PRIMARY KEY, name TEXT, region TEXT);
CREATE TABLE products_sales (product_id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL);
CREATE TABLE transactions_sales (
    transaction_id INTEGER PRIMARY KEY, customer_id INTEGER, product_id INTEGER,
    quantity INTEGER, transaction_date TEXT
);
''')

regions = ["Lagos", "Abuja", "Kano", "Ibadan", "Port Harcourt"]
categories = ["Electronics", "Groceries", "Fashion", "Home", "Sports"]

customers = [(i, f"Customer {i}", random.choice(regions)) for i in range(1, 121)]
products = [(i, f"Product {i}", random.choice(categories), round(random.uniform(5, 500), 2)) for i in range(1, 61)]
conn.executemany("INSERT INTO customers_sales VALUES (?,?,?)", customers)
conn.executemany("INSERT INTO products_sales VALUES (?,?,?,?)", products)

start = date(2026, 1, 1)
txns = [(tid, random.randint(1, 120), random.randint(1, 60), random.randint(1, 5),
         (start + timedelta(days=random.randint(0, 240))).isoformat()) for tid in range(1, 601)]
conn.executemany("INSERT INTO transactions_sales VALUES (?,?,?,?,?)", txns)
conn.commit()

total_revenue = conn.execute('''
    SELECT ROUND(SUM(t.quantity * p.price), 2) FROM transactions_sales t
    JOIN products_sales p ON p.product_id = t.product_id
''').fetchone()[0]
print("Total revenue:", total_revenue)

print("\nTop 5 customers by spend:")
for r in conn.execute('''
    SELECT c.name, ROUND(SUM(t.quantity*p.price),2) AS spend
    FROM customers_sales c JOIN transactions_sales t ON t.customer_id=c.customer_id
    JOIN products_sales p ON p.product_id=t.product_id
    GROUP BY c.customer_id ORDER BY spend DESC LIMIT 5
'''):
    print(dict(r))

## 17. ETL Capstone Pipeline (Part 70)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

def build_analytics_table(connection):
    connection.executescript('''
        DROP TABLE IF EXISTS revenue_by_month;
        CREATE TABLE revenue_by_month AS
        SELECT strftime('%Y-%m', transaction_date) AS month,
               ROUND(SUM(t.quantity * p.price), 2) AS revenue
        FROM transactions_sales t JOIN products_sales p ON p.product_id = t.product_id
        GROUP BY month;
    ''')
    logging.info("Analytics table 'revenue_by_month' built.")

build_analytics_table(conn)
report = pd.read_sql("SELECT * FROM revenue_by_month ORDER BY month", conn)
print(report)

## 18. SQLite + FTS5 RAG Retrieval Demo (Part 54)

This shows the retrieval half of a local RAG pipeline. The `llm_call` argument is left as a stub - plug in any Claude/LLM API call.

In [ ]:
def retrieve(query, top_k=3):
    try:
        cur = conn.execute('''
            SELECT title, content, bm25(documents_fts) AS score
            FROM documents_fts WHERE documents_fts MATCH ? ORDER BY score LIMIT ?
        ''', (query, top_k))
        return cur.fetchall()
    except sqlite3.OperationalError as e:
        print("FTS5 not available:", e)
        return []

def answer_question(question, llm_call=None):
    hits = retrieve(question)
    context = "\n\n".join(f"{r['title']}: {r['content']}" for r in hits)
    prompt = f"Answer using only this context:\n{context}\n\nQuestion: {question}"
    if llm_call is None:
        return prompt  # just show the assembled prompt if no LLM function is supplied
    return llm_call(prompt)

print(answer_question("search speed"))
# Note: FTS5 parses the MATCH argument itself as a small query language (words/phrases/operators),
# so raw punctuation like a trailing "?" in a full sentence can raise "fts5: syntax error near ?".
# Strip punctuation or quote the phrase before passing user text straight into MATCH.

## 19. Quick Reference - Close the Connection

See the full handbook (docx) for the complete cheat sheet, 100 interview questions, 75 SQL challenges, and 25 Python exercises.

In [ ]:
conn.execute("PRAGMA optimize;")
conn.close()
print("Connection closed. handbook.db retained on disk for further exploration.")